# G12 GPU path: OLMoE W4A16 (llm-compressor GPTQ) served by vLLM, scored on eval_mini

Kaggle: Accelerator **GPU T4 x2**, Internet **on**. Colab: a GPU runtime (T4 works; L4 or A100 is faster).

1. Start with `LIMIT = 5` and `SMOKE = True`: run all cells interactively (about 1 h; most of it is GPTQ).
2. For the real run set `LIMIT = None`, `SMOKE = False` and use **Save Version → Save & Run All (Commit)**.
   The results zip is in that version's Output tab; unzip it into `teams/12-quantization/gpu_path/results/`.

Steps: download the bf16 checkpoint at the pinned revision → check it → quantize to W4A16 → check that the router
and `lm_head` stayed BF16 and every expert was quantized → score bf16 and W4A16 with the same vLLM settings →
paired comparison → `runs.csv` rows. Each step stops the run if it fails. Weights stay in the temp folder; only
their SHA-256 manifests go into the zip.

In [ ]:
# ---- settings ----
LIMIT = 5                 # items per task; None = the full eval_mini profile (1,870 items)
SMOKE = True              # True: 32 calibration samples (tests the pipeline); False: 256 x 1024 tokens
METHOD = "gptq"           # "gptq", or "rtn" (no calibration) if GPTQ fails on an expert
FORMATS = ["bf16", "W4A16"]
RUN_BY = "SomaKorada07"
REPO_URL = "https://github.com/SomaKorada07/era-v5-lab.git"
BRANCH = "g12-week2-soma"
MODEL = "allenai/OLMoE-1B-7B-0924"
REVISION = "6d84c48581ece794365f2b8e9cfb043c68ade9c5"
# Unpinned for the first run: the versions used are printed and saved in environment.json; pin them after.
QUANT_PKGS = ["llmcompressor", "datasets"]
EVAL_PKGS = ["vllm", "lm_eval==0.4.13", "datasets==5.1.0"]

In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working" if KAGGLE else "/content")
TMP = Path("/kaggle/tmp" if KAGGLE else "/content/tmp")      # big files, outside the 20 GB Kaggle output
OUT = WORK / "gpu_path_out"; RUNS = OUT / ("runs_limit%s" % LIMIT if LIMIT else "runs")
for d in (TMP, OUT, RUNS): d.mkdir(parents=True, exist_ok=True)

def sh(cmd, **kw):
    print("$", cmd, flush=True)
    p = subprocess.run(cmd, shell=True, text=True, **kw)
    if p.returncode: raise SystemExit(f"failed ({p.returncode}): {cmd}")

REPO = TMP / "era-v5-lab"
if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
GP = REPO / "teams/12-quantization/gpu_path"
G9 = REPO / "teams/09-evaluation/harness/run_eval_mini.py"
gpus = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
                      shell=True, capture_output=True, text=True).stdout.strip()
print(gpus)
N_GPU = len(gpus.splitlines())
TP = 2 if N_GPU >= 2 else 1     # bf16 OLMoE (13.8 GB) does not fit one 16 GB T4 with a KV cache
env = {"started": time.strftime("%Y-%m-%dT%H:%M:%S"), "platform": "kaggle" if KAGGLE else "colab",
       "run_by": RUN_BY, "gpu": gpus, "tp": TP,
       "settings": dict(LIMIT=LIMIT, SMOKE=SMOKE, METHOD=METHOD, FORMATS=FORMATS, MODEL=MODEL, REVISION=REVISION,
                        BRANCH=BRANCH)}

In [ ]:
# two separate environments: llm-compressor and vLLM each pin their own torch and transformers
sh("pip -q install uv")
for name, pkgs in (("quant", QUANT_PKGS), ("eval", EVAL_PKGS)):
    venv = TMP / f"venv-{name}"
    if not (venv / "bin/python").exists():
        sh(f"uv venv -q --python 3.12 {venv}")
        sh(f"uv pip install -q --python {venv}/bin/python " + " ".join(f"'{p}'" for p in pkgs))
QPY, EPY = TMP / "venv-quant/bin/python", TMP / "venv-eval/bin/python"
versions = {}
for name, py, mods in (("quant", QPY, "llmcompressor compressed_tensors transformers torch datasets"),
                       ("eval", EPY, "vllm lm_eval transformers torch datasets")):
    code_ = "import importlib.metadata as m; print({k: m.version(k.replace('_','-')) for k in '%s'.split()})" % mods
    versions[name] = subprocess.run([str(py), "-c", code_], capture_output=True, text=True, check=True).stdout.strip()
    print(name, versions[name])
env["packages"] = versions

In [ ]:
# bf16 checkpoint at the pinned revision
BF16 = TMP / "OLMoE-bf16"
if not (BF16 / "config.json").exists():
    sh(f"{QPY} -c \"from huggingface_hub import snapshot_download as s; "
       f"s('{MODEL}', revision='{REVISION}', local_dir='{BF16}', allow_patterns=['*.json','*.safetensors','*.txt'])\"")
sh(f"python {GP}/model_check.py {BF16} --label bf16 --out {OUT}/manifest_bf16.json")

In [ ]:
W4 = TMP / (f"OLMoE-W4A16-{METHOD}" + ("-smoke" if SMOKE else ""))   # a smoke model is never reused for the real run
if not (W4 / "quantize_meta.json").exists():
    samples = "--samples 32 --seq-len 512" if SMOKE else "--samples 256 --seq-len 1024"
    sh(f"{QPY} {GP}/quantize_w4a16.py --model-dir {BF16} --out {W4} --method {METHOD} {samples}")
# fails unless the router and lm_head are BF16/F32 and every expert and attention projection is 4-bit
sh(f"python {GP}/model_check.py {W4} --label W4A16 --base {BF16} --expect-quantized --out {OUT}/manifest_W4A16.json")

In [ ]:
# same vLLM settings for both formats; each in its own process so the GPU memory is freed between them
DIRS = {"bf16": BF16, "W4A16": W4}
lim = f"--limit {LIMIT}" if LIMIT else ""
for fmt in FORMATS:
    t = time.time()
    sh(f"set -o pipefail; {EPY} {GP}/run_mini_vllm.py --model-dir {DIRS[fmt]} --format {fmt} --out {RUNS} --g9-harness {G9} "
       f"--tp {TP} {lim} 2>&1 | tee -a {OUT}/eval_{fmt}.log", executable="/bin/bash")
    env.setdefault("formats", {})[fmt] = {"wall_clock_s": round(time.time() - t)}

In [ ]:
sh(f"python {REPO}/teams/12-quantization/eval_mini/paired.py {RUNS} --base bf16 --out {RUNS}/paired.md")
print(open(f"{RUNS}/paired.md").read())
if LIMIT is None:
    sh(f"python {GP}/add_runs_rows.py --runs {RUNS} --manifests {OUT}/manifest_bf16.json {OUT}/manifest_W4A16.json "
       f"> {OUT}/runs_rows.csv")
    print(open(f"{OUT}/runs_rows.csv").read())
shutil.copy(W4 / "quantize_meta.json", OUT / "quantize_meta.json")
env["finished"] = time.strftime("%Y-%m-%dT%H:%M:%S")
(OUT / "environment.json").write_text(json.dumps(env, indent=1))
zip_path = shutil.make_archive(str(WORK / "gpu_path_results"), "zip", OUT)
print("results:", zip_path)